# Plant Doctor — GPU training notebook (Google Colab)

This notebook trains the Plant Doctor EfficientNet-B0 model on the PlantVillage dataset using a free Colab GPU and writes the finished model files back to your Google Drive.

**How to use (no code changes needed):**
1. Put the `plant-doctor` project folder in your Google Drive at `MyDrive/plant-doctor` (the whole folder, including `training/`, `models/`, `config.py`).
2. In Colab choose **Runtime → Change runtime type → T4 GPU** (or any GPU).
3. Choose **Runtime → Run all**. Approve the Google Drive access prompt when it appears.
4. Wait (about 25–45 minutes on a T4). When it finishes, the trained model and evaluation report are in `MyDrive/plant-doctor/artifacts/`.

Everything below is automatic.

In [ ]:
#@title 1. Settings (defaults work; change only if your folder is elsewhere)
PROJECT_DIR_IN_DRIVE = "plant-doctor"   #@param {type:"string"}
EPOCHS = 6                              #@param {type:"integer"}
BATCH_SIZE = 64                         #@param {type:"integer"}
VAL_FRACTION = 0.10                     #@param {type:"number"}
SEED = 42                               #@param {type:"integer"}
import os, sys, subprocess, shutil, pathlib, time
print("Settings OK")

In [ ]:
#@title 2. Check the GPU and install dependencies
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (training will be slow)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.23", "scikit-learn", "matplotlib", "pillow", "scipy", "reportlab", "gradio>=6,<7"], check=True)
print("Dependencies installed")

In [ ]:
#@title 3. Connect Google Drive and copy the project into the Colab workspace
from google.colab import drive
drive.mount('/content/drive')
import shutil, pathlib
src = pathlib.Path('/content/drive/MyDrive') / PROJECT_DIR_IN_DRIVE
assert src.exists(), f"Project folder not found at {src}. Upload the plant-doctor folder to MyDrive first."
dst = pathlib.Path('/content/plant-doctor')
if dst.exists(): shutil.rmtree(dst)
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('.venv', '__pycache__', '*.pt', 'last_checkpoint.pt'))
print("Project copied to", dst)

In [ ]:
#@title 4. Download the PlantVillage dataset (about 2 GB, 3–6 minutes)
from huggingface_hub import hf_hub_download
import zipfile, pathlib
data_root = pathlib.Path('/content/datasets/plantvillage'); (data_root / 'raw').mkdir(parents=True, exist_ok=True)
repo = "mohanty/PlantVillage"
for f in ["splits/color_train.txt", "splits/color_test.txt", "leaf_grouping/leaf-map.json"]:
    p = hf_hub_download(repo, f, repo_type="dataset")
    target = data_root / 'raw' / f; target.parent.mkdir(parents=True, exist_ok=True); shutil.copy(p, target)
zip_path = hf_hub_download(repo, "data.zip", repo_type="dataset")
with zipfile.ZipFile(zip_path) as z:
    members = [n for n in z.namelist() if n.startswith('raw/color/') and not n.endswith('/')]
    z.extractall(data_root, members=members)
n = sum(1 for _ in (data_root / 'raw' / 'color').rglob('*.*'))
print(f"Extracted {n} colour images to {data_root}")
assert n > 50000, "Dataset extraction looks incomplete"


In [ ]:
#@title 5. Train (GPU). Progress is printed every 25 steps; each epoch ends with validation metrics.
import subprocess, sys, os
os.chdir('/content/plant-doctor')
cmd = [sys.executable, "-m", "training.train", "--data-root", "/content/datasets/plantvillage", "--epochs", str(EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--workers", "4", "--val-fraction", str(VAL_FRACTION), "--seed", str(SEED), "--patience", "2"]
print(" ".join(cmd)); r = subprocess.run(cmd); assert r.returncode == 0, "Training failed — scroll up for the error"
print("Training finished. Checkpoint:", os.path.exists('artifacts/plant_doctor_efficientnet_b0.pt'))

In [ ]:
#@title 6. Evaluate on the held-out test split and fit the novelty detector (5–10 minutes)
import subprocess, sys, os
os.chdir('/content/plant-doctor')
r = subprocess.run([sys.executable, "-m", "training.evaluate", "--workers", "4"]); assert r.returncode == 0, "Evaluation failed"
print(open('artifacts/evaluation_report.md').read()[:3000])

In [ ]:
#@title 7. Copy the finished model files back to Google Drive
import shutil, pathlib
src = pathlib.Path('/content/plant-doctor/artifacts'); dst = pathlib.Path('/content/drive/MyDrive') / PROJECT_DIR_IN_DRIVE / 'artifacts'
dst.mkdir(parents=True, exist_ok=True)
for p in src.iterdir():
    if p.name in ('last_checkpoint.pt', 'splits'): continue
    if p.is_file(): shutil.copy(p, dst / p.name)
print("Copied to", dst); print(sorted(x.name for x in dst.iterdir()))
print("\nDone. Download the plant-doctor folder from Drive (or sync it) and run run_plant_doctor.bat on your computer.")